In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.kuzmin.lesson3 import Exercise, Layer, Loss

%load_ext autoreload
%autoreload 2

In [ ]:
X, y = load_digits(return_X_y=True)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)


def normalise(data):
    return 2 * data / np.max(data) - 1


def reset_and_train(
    X_train: np.ndarray,
    y_train: np.ndarray,
    n_neurons: int,
    lr: float,
    n_epoch: int,
    batch_size: int,
    loss_fn: Loss,
) -> Layer:
    rng = np.random.default_rng(42)

    model = Exercise.create_model(
        Exercise.create_linear_layer(64, n_neurons, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(n_neurons, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )

    Exercise.train_model(model, loss_fn, X_train, y_train, lr, n_epoch, batch_size)

    return model


X_train = normalise(X_train)
X_val = normalise(X_val)
X_train_val = normalise(X_train_val)
X_test = normalise(X_test)


def train():

    rng = np.random.default_rng()
    lr = rng.uniform(1e-6, 1e-2)
    n_epoch = rng.integers(5, 50, dtype=int)
    batch_size = rng.integers(1, X_train.shape[0] + 1, dtype=int)
    n_neurons = rng.integers(5, 500, dtype=int)
    loss_fn = Exercise.create_nll_loss()

    model = reset_and_train(X_train, y_train, n_neurons, lr, n_epoch, batch_size, loss_fn)

    accuracy = (np.argmax(model.forward(X_val), axis=1) == y_val).mean()

    return accuracy, {
        "n_neurons": n_neurons,
        "lr": lr,
        "n_epoch": n_epoch,
        "batch_size": batch_size,
    }

In [ ]:
best_acc = -1
best_params = {}
for _ in range(200):
    accuracy, params = train()
    if best_acc < accuracy:
        best_acc = accuracy
        best_params = params
        print(f"Current accuracy: {best_acc}, model = {best_params}")

In [ ]:
import pandas as pd


def make_cm(p, y):
    cm = np.zeros((10, 10), dtype=int)

    for true, pred in zip(y, p, strict=True):
        cm[true, pred] += 1

    return cm


loss_fn = Exercise.create_nll_loss()
model = reset_and_train(
    X_train_val,
    y_train_val,
    best_params["n_neurons"],
    best_params["lr"],
    best_params["n_epoch"],
    best_params["batch_size"],
    loss_fn,
)

p = np.argmax(model.forward(X_test), axis=1)
test_acc = (p == y_test).mean()
print(f"Trained model. True accuracy on test data = {test_acc} on model with parameters {best_params}.")

cm = make_cm(p, y_test)
cm = pd.DataFrame(
    cm,
    index=[f"True_{i}" for i in range(10)],
    columns=[f"Pred_{i}" for i in range(10)],
)
cm